# Kalman Filter — Synthetic Signal Validation

Before applying any filter to real market data it is essential to test it on
a signal where the **true underlying value is known**. Here we:

1. Generate a synthetic random-walk process (the true price) and corrupt it with
   Gaussian noise (simulated bid-ask noise / market microstructure).
2. Run `KalmanFilter1D` with several Q/R choices and compare how well each
   recovers the true signal.
3. Run `KalmanFilter2D` on a signal with a known trend to confirm the filter
   extracts the trend component correctly.
4. Quantify performance with the metrics in `metrics.py`.


In [ ]:
import sys
from pathlib import Path

# notebooks/ is one level inside project/ — add project/ to path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kalman_filter import KalmanFilter1D, KalmanFilter2D
import metrics

plt.rcParams['figure.dpi'] = 110
print('Imports OK')


In [ ]:
# ── Synthetic random walk ──────────────────────────────────────────────────
rng = np.random.default_rng(42)
N       = 400
Q_TRUE  = 0.25   # daily variance of the true random walk
R_TRUE  = 4.0    # variance of the measurement noise (std ≈ 2.0)

true_price = np.zeros(N)
for i in range(1, N):
    true_price[i] = true_price[i-1] + rng.standard_normal() * Q_TRUE**0.5

measurements = true_price + rng.standard_normal(N) * R_TRUE**0.5

print(f'Signal std (increments):  {np.std(np.diff(true_price)):.3f}')
print(f'Measurement noise std:    {np.std(measurements - true_price):.3f}')
print(f'SNR (dB): {20*np.log10(np.std(true_price)/np.std(measurements-true_price)):.1f}')


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(measurements, color='#9E9E9E', lw=0.8, alpha=0.7, label='Noisy measurements')
ax.plot(true_price,   color='#2196F3', lw=1.8,             label='True signal')
ax.set_title('Synthetic Random Walk: True Signal vs Noisy Measurements', fontsize=12)
ax.set_xlabel('Time (samples)')
ax.set_ylabel('Value')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 1-D Kalman Filter — Effect of Process Noise Q

The ratio **Q/R** controls the filter's agility:

| Q/R | Behaviour |
|-----|-----------|
| small | Trusts the model: smooth estimate, but slow to react to real changes |
| matched | Optimal when Q and R equal the true noise levels |
| large | Trusts the measurements: responsive, but noisy |


In [ ]:
configs = [
    {'Q': 0.01,  'color': '#4CAF50', 'label': 'Q=0.01 (smooth / lagging)'},
    {'Q': 0.25,  'color': '#FF5722', 'label': 'Q=0.25 (matched to truth)'},
    {'Q': 4.0,   'color': '#9C27B0', 'label': 'Q=4.0  (responsive / noisy)'},
]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(measurements, color='#9E9E9E', lw=0.7, alpha=0.45, label='Measurements', zorder=2)
ax.plot(true_price,   color='#2196F3', lw=2.2,             label='True signal',  zorder=5)

kf_results = {}
for cfg in configs:
    kf  = KalmanFilter1D(Q=cfg['Q'], R=R_TRUE, x0=measurements[0])
    est, gains, _ = kf.filter(measurements)
    kf_results[cfg['label']] = est
    ax.plot(est, color=cfg['color'], lw=1.4, label=cfg['label'], zorder=4)

ax.set_title('KalmanFilter1D — Q/R Tuning (R fixed at true value)', fontsize=12)
ax.set_xlabel('Time (samples)')
ax.set_ylabel('Value')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
rows = [metrics.summarize('Raw measurements', true_price, measurements)]
for label, est in kf_results.items():
    rows.append(metrics.summarize(f'KF1D  {label}', true_price, est))

df_metrics = pd.DataFrame(rows).set_index('Filter')
df_metrics


## 2-D Kalman Filter — Price + Trend

The 2-D filter adds a *trend state* (local velocity) alongside price.  This lets
the filter predict where the price is *going*, not just where it has been:

```
x = [price, trend]ᵀ
F = [[1, 1],      # price_{k+1} = price_k + trend_k + noise
     [0, 1]]      # trend_{k+1} = trend_k            + noise
H = [[1, 0]]      # we observe only price
```

We generate a signal with a known true trend of **+0.3 per sample** so we can
verify that the filter recovers it.


In [ ]:
TRUE_TREND  = 0.3
trend_prices = np.zeros(N)
for i in range(1, N):
    trend_prices[i] = trend_prices[i-1] + TRUE_TREND + rng.standard_normal() * 0.3
trend_meas = trend_prices + rng.standard_normal(N) * 3.0

Q2  = np.diag([0.1, 0.01])   # [price process noise, trend process noise]
R2  = 9.0                     # measurement noise variance
kf2 = KalmanFilter2D(Q=Q2, R=R2, x0=np.array([trend_meas[0], 0.0]))
prices2d, trends2d, _ = kf2.filter(trend_meas)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(trend_meas,   color='#9E9E9E', lw=0.8, alpha=0.55, label='Measurements')
ax1.plot(trend_prices, color='#2196F3', lw=1.8,             label='True price')
ax1.plot(prices2d,     color='#FF5722', lw=1.5, ls='--',    label='KF2D price estimate')
ax1.set_ylabel('Value')
ax1.set_title('KalmanFilter2D — Price Estimate and Extracted Trend Component', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(trends2d, color='#4CAF50', lw=1.5, label='Estimated trend')
ax2.axhline(TRUE_TREND, color='#2196F3', lw=1.5, ls='--',
            label=f'True trend = {TRUE_TREND}')
ax2.set_ylabel('Trend (units/sample)')
ax2.set_xlabel('Time (samples)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Final trend estimate: {trends2d[-1]:.4f}  (true = {TRUE_TREND})')


In [ ]:
# 1D vs 2D on the same trending signal
kf1_trend = KalmanFilter1D(Q=0.25, R=9.0, x0=trend_meas[0])
est1d, _, _ = kf1_trend.filter(trend_meas)

rows2 = [
    metrics.summarize('Raw measurements',        trend_prices, trend_meas),
    metrics.summarize('KalmanFilter1D  Q=0.25',  trend_prices, est1d),
    metrics.summarize('KalmanFilter2D',           trend_prices, prices2d),
]
pd.DataFrame(rows2).set_index('Filter')
